In [2]:
from openai import OpenAI
import gradio as gr
import os
import requests
import pandas as pd
import io
from urllib.parse import urlparse
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
from firecrawl import FirecrawlApp
fc_key = os.getenv('FC_API_KEY')
print(fc_key)
app = FirecrawlApp(api_key=fc_key)


# Initialize API clients
api_key = os.getenv('OPENAI_API_KEY')
#jina_api_key = os.getenv('JINA_API_KEY')
client = OpenAI(api_key=api_key)

fc-cfb1126cd23048f2801c2ab03115026f


In [3]:
def scrape_website(url):
    """
    Scrapes a website by scrolling down multiple times with delays.
    
    Args:
        url (str): The URL of the website to scrape
        
    Returns:
        str: The scraped content in markdown format
    """
    scrape_status = app.scrape_url(
        url,
        params={
            'formats': ['markdown'],
            'actions': [
                {"type": "scroll", "direction": 'down'},
                {"type": "wait", "milliseconds": 2000},
                {"type": "scroll", "direction": 'down'}, 
                {"type": "wait", "milliseconds": 2000},
                {"type": "scroll", "direction": 'down'},
                {"type": "wait", "milliseconds": 2000},
                {"type": "scroll", "direction": 'down'},
                {"type": "wait", "milliseconds": 2000},
            ]
        }
    )
    return scrape_status


def create_excel_list(url, user_prompt):
    """
    Create an Excel file from web content based on user prompt.
    
    Args:
        url: Target website URL
        user_prompt: Instructions for data extraction
        
    Returns:
        tuple: (markdown_content, status_message)
    """
    # Scrape web content
    raw_obj = scrape_website(url)
    raw_data = raw_obj['markdown']

    # Prepare GPT prompt
    system_prompt = """You are an assistant that extracts information from a webpage and creates a CSV formatted list of the information. 
    The exact instructions regarding the data needed to be extracted along with the raw data will be provided by the user prompt. For each kind of information, you will create a column. 
    You will only return well-formed CSV with columns separated by commas and each row on a new line. No Markdown tables, no additional text, no code fences."""

    # Get structured data from GPT
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"The user prompt is: {user_prompt}. The raw data is: {raw_data}"}
        ],
        temperature=0.0
    )

    csv_content = response.choices[0].message.content
    return save_to_excel(csv_content, url, raw_obj['metadata']['title'])

def save_to_excel(csv_content, url, page_title):
    """
    Save CSV content to Excel file with proper error handling.
    
    Returns:
        tuple: (markdown_content, status_message)
    """
    file_saved = False
    filename = f"{urlparse(url).netloc}_list.xlsx"
    sheet_name = sanitize_sheet_name(page_title)
    error_msg = ""
    
    try:
        df = pd.read_csv(io.StringIO(csv_content), sep=",", engine="python")
        sheet_name = handle_existing_file(filename, sheet_name, df)
        file_saved = True
        
    except pd.errors.EmptyDataError:
        error_msg = "Error: No data could be parsed from the response"
    except Exception as e:
        error_msg = f"Error: {str(e)}"

    status = generate_status_message(file_saved, filename, sheet_name, error_msg)
    return f"{status}\n\nExtracted Data:\n{csv_content}", status

def sanitize_sheet_name(title):
    """Create valid Excel sheet name from title."""
    sheet_name = title.strip()
    invalid_chars = [':', '\\', '/', '?', '*', '[', ']']
    for char in invalid_chars:
        sheet_name = sheet_name.replace(char, '')
    return sheet_name[:27] or "Sheet1"

def handle_existing_file(filename, sheet_name, df):
    """Handle writing to existing Excel file with sheet name conflicts."""
    if os.path.exists(filename):
        try:
            existing_wb = pd.read_excel(filename, sheet_name=None)
            sheet_name = get_unique_sheet_name(sheet_name, existing_wb.keys())
            with pd.ExcelWriter(filename, engine='openpyxl', mode='a') as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        except Exception:
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    return sheet_name

def get_unique_sheet_name(base_name, existing_names):
    """Generate unique sheet name avoiding conflicts."""
    test_name = base_name
    counter = 1
    while test_name in existing_names:
        test_name = f"{base_name}_{counter}"
        counter += 1
    return test_name

def generate_status_message(file_saved, filename, sheet_name, error_msg):
    """Generate appropriate status message based on operation result."""
    if file_saved:
        status = f"Success! Excel file saved as: {filename} (Sheet: {sheet_name})"
        if error_msg:
            status += f"\nNote: {error_msg}"
    else:
        status = f"Failed to save Excel file. {error_msg}"
    return status

In [4]:
url='https://hrpro.gr/hr-100-powerlist-2023/'
prompt='Grab a list with the names, companies and Linkedin profile links of all the people on the page.'
create_excel_list(url, prompt)

('Failed to save Excel file. Error: Expected 3 fields in line 23, saw 4\n\nExtracted Data:\nΔήμητρα Καλαμπαλίκη,AB Vassilopoulos,LinkedIn  \nΝικόλας Μοσχάτος,Accenture (Greece & Bulgaria),LinkedIn  \nΕμμανουέλα Καραμαλή,AEGEAN,LinkedIn  \nΧαράλαμπος Σιδηρόπουλος,Agris SA,LinkedIn  \nΒασιλική Μαρίνου,Alumil Group NEW,LinkedIn  \nΒάγια Αποστολίδη,Alumil Group,LinkedIn  \nΠόπη Αντωνοπούλου,Ambience Services,LinkedIn  \nΜαίρη Καλογεροπούλου,Ambience Services,LinkedIn  \nΣπυρίδων Χονδρογιάννης,Amplus S.A.,LinkedIn  \nΜάριος Θεοδωράτος,Angelicoussis Group,LinkedIn  \nΘάλεια Ανδριοπούλου,Atos Greece,LinkedIn  \nΒασίλης Χουλιάρας,Barilla Hellas,LinkedIn  \nΧριστίνα Σταύρου,Barilla Hellas,LinkedIn  \nΣτέργιος Λαψάνας,BASF Hellas,LinkedIn  \nTaleen Tchalikian,Celestyal Cruises,LinkedIn  \nΚώστας Βαβαρούτας,Cenergy Holdings,LinkedIn  \nΠόλυ Πάνου,Cenergy Holdings,LinkedIn  \nΚώστας Τσαλίκης,Citi Ελλάδας,LinkedIn  \nΚωνσταντίνα Μπάρκα,Coca Cola HBC,LinkedIn  \nΌλγα Κωνσταντίνου,Coca cola HBC,Linke